# HER2 nanobody — ESMC embedding on a free Colab T4

This notebook runs **only the expensive step**: one forward pass of frozen
`biohub/ESMC-300M` over all 19,367 sequences, cached to Google Drive. Everything
afterwards (training the classifier and the regressor) takes seconds on CPU and is
meant to run locally.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

### On confidentiality

The dataset is confidential. This notebook keeps it in **your own Drive** and never
writes it anywhere public — no GitHub push, no model hub, no shared notebook link.
Keep this notebook private, and delete `HER2_DIR` from Drive when the test is over.

### Why Drive

A free Colab session disconnects after a few hours and loses its local disk. The
embedding cache is the only artefact worth hours, so it is written straight to
Drive: a disconnect then costs a reconnect rather than the whole run.

In [ ]:
# 1. Confirm we actually have a GPU. On CPU this run takes hours, not minutes.
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

import torch

assert torch.cuda.is_available(), (
    "No CUDA device. Runtime -> Change runtime type -> T4 GPU, then rerun."
)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive and lay out the working directory.
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

HER2_DIR = Path("/content/drive/MyDrive/her2")
DATA_DIR = HER2_DIR / "data"
EMBED_DIR = HER2_DIR / "embeddings"
CODE_DIR = HER2_DIR / "code"
for directory in (DATA_DIR, EMBED_DIR, CODE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("data   ->", DATA_DIR)
print("cache  ->", EMBED_DIR)

## 3. Get the data and the code into Drive

Do this **once**. Two options:

* **Upload by hand** (simplest): open the Files pane in Colab, navigate to
  `drive/MyDrive/her2/data`, and drop in `train.csv`, `val.csv`, `test_public.csv`.
  Copy `embed_esm.py` and the `src/` folder into `drive/MyDrive/her2/code`.
* **Or run the cell below** after uploading a zip of the submission folder.

Do not `git clone` this from a public repository — the data is confidential.

In [ ]:
# Optional: unpack an uploaded zip of the submission folder into Drive.
# Skip if you uploaded the files by hand.
import zipfile

ZIP = Path("/content/submission.zip")  # upload via the Files pane, then run
if ZIP.exists():
    with zipfile.ZipFile(ZIP) as archive:
        archive.extractall(CODE_DIR)
    print("extracted to", CODE_DIR)
else:
    print(f"{ZIP} not found - assuming you uploaded the files by hand.")

missing = [
    name
    for name in ("train.csv", "val.csv", "test_public.csv")
    if not (DATA_DIR / name).exists()
]
assert not missing, f"still missing from {DATA_DIR}: {missing}"
print("data files present")

In [ ]:
# 4. Install ESM. The `esm` package is what registers the custom `esmc`
#    architecture with transformers; without it from_pretrained cannot resolve
#    biohub/ESMC-300M. Colab already has a suitable torch, so we do not touch it.
%pip install -q "esm@git+https://github.com/Biohub/esm.git@main"
%pip install -q transformers pandas numpy scikit-learn scipy wandb

In [ ]:
# 5. Smoke test: 32 sequences. Confirms the install resolves and the model loads.
%cd {CODE_DIR}
!python embed_esm.py --data-dir {DATA_DIR} --outdir /content/smoke --limit 32

## 6. The pooling check — run this before the full pass

The tokenizer adds BOS/EOS and pads each batch to its longest member. Mean-pooling
the raw tensor averages in that padding, so a sequence's vector silently depends on
which other sequences shared its batch: everything still runs, the numbers are just
worse and irreproducible.

`--check` embeds one sequence alone and again inside a batch of 32 mixed lengths and
compares. They must match to floating-point tolerance.

In [ ]:
!python embed_esm.py --data-dir {DATA_DIR} --check

In [ ]:
# 7. The full pass: 19,367 sequences, ~15 minutes on a T4.
#
#    Several layers are cached in one pass so Task 1 can choose on validation
#    without paying for ESMC again -- the last layer of a masked-LM is specialised
#    for token reconstruction and is often not the best sentence representation.
#    Writing straight to Drive means a disconnect costs a reconnect, not the run.
!python embed_esm.py \
    --data-dir {DATA_DIR} \
    --outdir {EMBED_DIR} \
    --layers 6,12,18,24,30 \
    --batch-size 32 \
    --seed 0

In [ ]:
# 8. Verify the cache: shapes, id coverage, and no NaNs.
import numpy as np
import pandas as pd

for split in ("train", "val", "test_public"):
    with np.load(EMBED_DIR / f"{split}.npz", allow_pickle=False) as blob:
        frame = pd.read_csv(DATA_DIR / f"{split}.csv", usecols=["id"])
        layers = sorted(k for k in blob.files if k.startswith("layer_"))
        matrix = blob[layers[-1]]
        assert set(frame.id) == set(blob["ids"].tolist()), f"{split}: id mismatch"
        assert np.isfinite(matrix).all(), f"{split}: non-finite values"
        print(f"{split:12s} {matrix.shape}  layers={[l.split('_')[1] for l in layers]}  OK")

## 9. Done — pull the cache down and train locally

`embeddings/` is roughly 370 MB for five layers. Download it (or keep working in
Drive) and then, locally:

```bash
uv run python train_task1.py --sweep-layers
uv run python train_task1.py --layer <best> --cv --wandb
uv run python train_task2.py --transforms
uv run python train_task2.py --model two_stage --compare-combines --wandb
```

Both training scripts run on CPU in seconds. **Do not re-run ESMC.**

If you would rather train here too, the cells below work unchanged — wandb will ask
for a login once.

In [ ]:
# Optional: train in Colab instead of locally.
# !wandb login
# !python train_task1.py --data-dir {DATA_DIR} --embed-dir {EMBED_DIR} --sweep-layers
# !python train_task1.py --data-dir {DATA_DIR} --embed-dir {EMBED_DIR} --layer 30 --cv --wandb
# !python train_task2.py --data-dir {DATA_DIR} --embed-dir {EMBED_DIR} --transforms
# !python train_task2.py --data-dir {DATA_DIR} --embed-dir {EMBED_DIR} --model two_stage --compare-combines --wandb